In [2]:
import pandas as pd 
import ast

df = pd.read_csv(r"D:\ML\ak_GPT\data\full_dataset.csv")


In [5]:
# Preprocess the data to create a single string for each recipe with special tokens

# Your special tokens
special_tokens = [
    "<|startofrecipe|>",
    "<|endofrecipe|>", 
    "<|title|>",
    "<|ingredients|>",
    "<|directions|>",
]

def process_row(row):
    title = row["title"]
    ingredients = ast.literal_eval(row["ingredients"])
    directions = ast.literal_eval(row["directions"])
    
    ingredients_str = " | ".join(ingredients)
    directions_str = " | ".join(directions)
    
    return (
        f"<|startofrecipe|> <|title|> {title} "
        f"<|ingredients|> {ingredients_str} "
        f"<|directions|> {directions_str} <|endofrecipe|>"
    )

df["processed"] = df.apply(process_row, axis=1)

with open(r"D:\ML\ak_GPT\data\processed_recipes.txt", "w", encoding="utf-8") as f:
    for recipe in df["processed"]:
        f.write(recipe + "\n")

In [6]:
# Display the first few processed recipes from the text file 

with open(r"D:\ML\ak_GPT\data\processed_recipes.txt", "r", encoding="utf-8") as f:
    for _ in range(5):
        print(f.readline())

<|startofrecipe|> <|title|> No-Bake Nut Cookies <|ingredients|> 1 c. firmly packed brown sugar | 1/2 c. evaporated milk | 1/2 tsp. vanilla | 1/2 c. broken nuts (pecans) | 2 Tbsp. butter or margarine | 3 1/2 c. bite size shredded rice biscuits <|directions|> In a heavy 2-quart saucepan, mix brown sugar, nuts, evaporated milk and butter or margarine. | Stir over medium heat until mixture bubbles all over top. | Boil and stir 5 minutes more. Take off heat. | Stir in vanilla and cereal; mix well. | Using 2 teaspoons, drop and shape into 30 clusters on wax paper. | Let stand until firm, about 30 minutes. <|endofrecipe|>

<|startofrecipe|> <|title|> Jewell Ball'S Chicken <|ingredients|> 1 small jar chipped beef, cut up | 4 boned chicken breasts | 1 can cream of mushroom soup | 1 carton sour cream <|directions|> Place chipped beef on bottom of baking dish. | Place chicken on top of beef. | Mix soup and cream together; pour over chicken. Bake, uncovered, at 275° for 3 hours. <|endofrecipe|>

<

In [14]:
import sentencepiece as spm

spm.SentencePieceTrainer.train(
    input=r"D:\ML\ak_GPT\data\processed_recipes.txt",
    model_prefix=r"D:\ML\ak_GPT\data\recipe_tokenizer",
    vocab_size=32000,
    model_type="bpe",
    pad_id=0,
    unk_id=1,
    bos_id=2,
    eos_id=3,
    user_defined_symbols=[
        "<|startofrecipe|>",
        "<|endofrecipe|>",
        "<|title|>",
        "<|ingredients|>",
        "<|directions|>",
    ],
    character_coverage=1.0,
    input_sentence_size=1_000_000,
    shuffle_input_sentence=True,
)

In [18]:
sp = spm.SentencePieceProcessor()
sp.load(r"D:\ML\ak_GPT\data\recipe_tokenizer.model")

# Check vocab size
print(f"Vocab size: {sp.get_piece_size()}")

# Check that special tokens are single tokens
for token in ["<|startofrecipe|>", "<|title|>", "<|ingredients|>", "<|directions|>", "<|endofrecipe|>"]:
    id = sp.piece_to_id(token)
    print(f"{token} → ID {id}")

# Encode a sample recipe snippet
test = "<|title|> Garlic Butter Pasta <|ingredients|> 2 cloves garlic | 3 Tbsp. butter"
tokens = sp.encode(test, out_type=str)
ids = sp.encode(test)
print(f"\nTokens: {tokens}")
print(f"IDs: {ids}")
print(f"Decoded: {sp.decode(ids)}")

Vocab size: 32000
<|startofrecipe|> → ID 4
<|title|> → ID 6
<|ingredients|> → ID 7
<|directions|> → ID 8
<|endofrecipe|> → ID 5

Tokens: ['▁', '<|title|>', '▁Garlic', '▁Butter', '▁Pasta', '▁', '<|ingredients|>', '▁2', '▁cloves', '▁garlic', '▁|', '▁3', '▁Tbsp', '.', '▁butter']
IDs: [31573, 6, 1635, 895, 2195, 31573, 7, 50, 593, 256, 9, 71, 326, 31591, 157]
Decoded: <|title|> Garlic Butter Pasta <|ingredients|> 2 cloves garlic | 3 Tbsp. butter
